# HSTU vs SASRec — cached streaming latency on A100

This is a **serving-only** companion to the HSTU canonical finalizer.

It reuses the validated SASRec architecture from the Sparse Walker benchmark:

- `d_model = 64`
- 2 blocks
- 2 attention heads
- FFN = 256
- dropout = 0.2 (disabled at inference)
- item + positional embeddings, **no `sqrt(d)` scaling**
- pre-LN causal self-attention
- final LayerNorm

The canonical Beauty model at `max_len=50`, `n_items=12,101` has exactly
**877,824 trainable parameters**.

For serving comparison we extend only the positional table to 211 positions so
we can benchmark the same history lengths as the HSTU finalizer. The Transformer
body is unchanged.

## Fair comparison

HSTU's current latency table measures:

> append one event to an existing cache → update the model state

Therefore this notebook measures the **same thing for SASRec**. It deliberately
excludes full-catalog scoring/top-k. Catalog scoring should be compared separately
with the same retrieval policy for both models.

The notebook:
1. validates full-history SASRec ↔ cached SASRec numerical parity;
2. benchmarks cached SASRec with PyTorch SDPA;
3. reads the HSTU latency rows already stored in `HSTU_CANONICAL_v1_stamp.json`;
4. prints direct SASRec/HSTU speed ratios.

In [ ]:
import math, json, time
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Use a GPU Colab runtime."

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

HSTU_STAMP = Path(
    "/content/drive/MyDrive/hstu_pure_pytorch_ml1m/HSTU_CANONICAL_v1_stamp.json"
)
assert HSTU_STAMP.exists(), (
    "Missing HSTU stamp. Run the HSTU finalizer first: " + str(HSTU_STAMP)
)

hstu_stamp = json.loads(HSTU_STAMP.read_text())
hstu_latency_df = pd.DataFrame(hstu_stamp["latency"])
print("Loaded HSTU stamp:", HSTU_STAMP)
print("HSTU stamp status:", hstu_stamp.get("status"))
display(hstu_latency_df.head())

## Exact SASRec architecture from the validated benchmark

In [ ]:
def init_embedding(emb):
    nn.init.normal_(emb.weight, mean=0.0, std=0.02)
    if emb.padding_idx is not None:
        with torch.no_grad():
            emb.weight[emb.padding_idx].zero_()


class FFN(nn.Module):
    def __init__(self, d, inner, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, inner),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(inner, d),
        )

    def forward(self, x):
        return self.net(x)


class SASBlock(nn.Module):
    def __init__(self, d, heads, inner, dropout):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(
            d, heads, dropout=dropout, batch_first=True
        )
        self.n2 = nn.LayerNorm(d)
        self.ffn = FFN(d, inner, dropout)
        self.d1 = nn.Dropout(dropout)
        self.d2 = nn.Dropout(dropout)

    def forward(self, x, padding=None):
        L = x.size(1)
        causal = torch.triu(
            torch.ones(L, L, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        z = self.n1(x)
        a, _ = self.attn(
            z, z, z,
            attn_mask=causal,
            key_padding_mask=padding,
            need_weights=False,
        )
        x = x + self.d1(a)
        x = x + self.d2(self.ffn(self.n2(x)))
        return x


class SASRec(nn.Module):
    def __init__(
        self,
        n_items=12_101,
        max_len=211,
        d=64,
        layers=2,
        heads=2,
        inner=256,
        dropout=0.2,
    ):
        super().__init__()
        self.n_items = n_items
        self.max_len = max_len
        self.d_model = d
        self.item = nn.Embedding(n_items + 1, d, padding_idx=0)
        init_embedding(self.item)
        self.pos = nn.Embedding(max_len, d)
        init_embedding(self.pos)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            SASBlock(d, heads, inner, dropout)
            for _ in range(layers)
        ])
        self.norm = nn.LayerNorm(d)

    def encode(self, seq):
        B, L = seq.shape
        p = torch.arange(L, device=seq.device)[None]
        x = self.drop(self.item(seq) + self.pos(p))
        padding = seq == 0
        for block in self.blocks:
            x = block(x, padding)
        return self.norm(x)

    def last_hidden(self, seq):
        return self.encode(seq)[:, -1]


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


canonical_50 = SASRec(max_len=50).to(DEVICE).eval()
assert n_params(canonical_50) == 877_824, n_params(canonical_50)
print("Canonical SASRec max_len=50 params:", n_params(canonical_50))

sas = SASRec(max_len=211).to(DEVICE).eval()
print("Serving SASRec max_len=211 params:", n_params(sas))
print("Extra params are positional embeddings only:",
      n_params(sas) - n_params(canonical_50))
del canonical_50

## Exact cached one-event SASRec

In [ ]:
@dataclass
class SASCache:
    k: list
    v: list
    lengths: torch.Tensor


def make_cache(model, batch_size, dtype=torch.float32):
    k, v = [], []
    for block in model.blocks:
        H = block.attn.num_heads
        D = model.d_model
        Dh = D // H
        k.append(torch.zeros(
            batch_size, H, model.max_len, Dh,
            device=DEVICE, dtype=dtype,
        ))
        v.append(torch.zeros(
            batch_size, H, model.max_len, Dh,
            device=DEVICE, dtype=dtype,
        ))
    lengths = torch.zeros(
        batch_size, device=DEVICE, dtype=torch.long
    )
    return SASCache(k=k, v=v, lengths=lengths)


def _project_qkv(mha, z):
    D = z.shape[-1]
    W = mha.in_proj_weight
    b = mha.in_proj_bias

    q = F.linear(z, W[:D], None if b is None else b[:D])
    k = F.linear(z, W[D:2*D], None if b is None else b[D:2*D])
    v = F.linear(z, W[2*D:], None if b is None else b[2*D:])

    H = mha.num_heads
    Dh = D // H
    q = q.view(z.size(0), H, Dh)
    k = k.view(z.size(0), H, Dh)
    v = v.view(z.size(0), H, Dh)
    return q, k, v


def _scatter_kv(cache_tensor, values, pos):
    B, H, _, Dh = cache_tensor.shape
    idx = pos[:, None, None, None].expand(B, H, 1, Dh)
    cache_tensor.scatter_(2, idx, values.unsqueeze(2))


@torch.inference_mode()
def sas_cached_step_sdpa(model, cache, item_ids, advance=True):
    # Append one event and compute the hidden state at the new position.
    pos = cache.lengths.clone()
    assert int(pos.max()) < model.max_len

    x = model.item(item_ids) + model.pos(pos)

    L = int(pos[0].item()) + 1
    if not bool(torch.all(pos == pos[0])):
        raise ValueError("Benchmark cache expects uniform lengths within a batch.")

    for li, block in enumerate(model.blocks):
        z = block.n1(x)
        q, k_new, v_new = _project_qkv(block.attn, z)

        _scatter_kv(cache.k[li], k_new, pos)
        _scatter_kv(cache.v[li], v_new, pos)

        a = F.scaled_dot_product_attention(
            q.unsqueeze(2),
            cache.k[li][:, :, :L, :],
            cache.v[li][:, :, :L, :],
            dropout_p=0.0,
            is_causal=False,
        ).squeeze(2)

        a = a.reshape(x.size(0), model.d_model)
        a = block.attn.out_proj(a)

        x = x + a
        x = x + block.ffn(block.n2(x))

    out = model.norm(x)
    if advance:
        cache.lengths.add_(1)
    return out

## Full-history ↔ cached parity

In [ ]:
@torch.inference_mode()
def parity_test(model, lengths=(1, 2, 5, 16, 50, 100, 200), seed=123):
    torch.manual_seed(seed)
    seq = torch.randint(
        1, model.n_items + 1,
        (1, max(lengths)),
        device=DEVICE,
    )

    cache = make_cache(model, batch_size=1)
    rows = []

    wanted = set(lengths)
    for t in range(seq.size(1)):
        cached = sas_cached_step_sdpa(
            model, cache, seq[:, t], advance=True
        )

        L = t + 1
        if L in wanted:
            full = model.last_hidden(seq[:, :L])
            err = float((cached - full).abs().max().cpu())
            cos = float(
                F.cosine_similarity(
                    cached.float(), full.float()
                ).item()
            )
            rows.append({
                "length": L,
                "max_abs": err,
                "cosine": cos,
            })

    return pd.DataFrame(rows)


parity_df = parity_test(sas)
display(parity_df)

worst = float(parity_df["max_abs"].max())
print("Worst full↔cached max abs:", worst)
assert worst < 5e-4, (
    "Cached SASRec parity failed; do not use latency result."
)
print("PARITY: PASS")

## Same CUDA timing protocol as HSTU

In [ ]:
BENCH_HISTORY = (16, 50, 100, 200)
BENCH_BATCHES = (1, 32, 128)
BENCH_WARMUP = 50
BENCH_ITERS = 300


def cuda_bench(fn, warmup=BENCH_WARMUP, iters=BENCH_ITERS):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    vals = []
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    for _ in range(iters):
        start.record()
        fn()
        end.record()
        end.synchronize()
        vals.append(start.elapsed_time(end))

    arr = np.asarray(vals)
    return {
        "mean_ms": float(arr.mean()),
        "p50_ms": float(np.percentile(arr, 50)),
        "p95_ms": float(np.percentile(arr, 95)),
        "p99_ms": float(np.percentile(arr, 99)),
    }


def seed_sas_cache(model, batch, history):
    assert history < model.max_len
    cache = make_cache(model, batch)

    for k, v in zip(cache.k, cache.v):
        k.normal_(0, 0.2)
        v.normal_(0, 0.2)

    cache.lengths.fill_(history)
    item = torch.randint(
        1, model.n_items + 1,
        (batch,),
        device=DEVICE,
        dtype=torch.long,
    )
    return cache, item


rows = []
for B in BENCH_BATCHES:
    for L in BENCH_HISTORY:
        cache, item = seed_sas_cache(sas, B, L)

        timing = cuda_bench(
            lambda: sas_cached_step_sdpa(
                sas, cache, item, advance=False
            )
        )

        row = {
            "variant": "SASRec",
            "backend": "cached_sdpa",
            "batch": B,
            "history": L,
            **timing,
        }
        rows.append(row)
        print(row)

sas_latency_df = pd.DataFrame(rows)
display(sas_latency_df)

## Direct HSTU / SASRec comparison

In [ ]:
h = hstu_latency_df[
    hstu_latency_df["backend"] == "cached_triton_attention"
].copy()

s = sas_latency_df.copy()

comp = h.merge(
    s,
    on=["batch", "history"],
    suffixes=("_hstu", "_sasrec"),
)

comp["sasrec_over_hstu"] = (
    comp["p50_ms_sasrec"] / comp["p50_ms_hstu"]
)
comp["hstu_over_sasrec"] = (
    comp["p50_ms_hstu"] / comp["p50_ms_sasrec"]
)

comp = comp[[
    "variant_hstu",
    "batch",
    "history",
    "p50_ms_sasrec",
    "p50_ms_hstu",
    "sasrec_over_hstu",
    "hstu_over_sasrec",
    "p95_ms_sasrec",
    "p95_ms_hstu",
]].rename(columns={
    "variant_hstu": "HSTU_variant",
    "p50_ms_sasrec": "SASRec_p50_ms",
    "p50_ms_hstu": "HSTU_p50_ms",
    "sasrec_over_hstu": "HSTU_speedup_vs_SASRec",
    "hstu_over_sasrec": "HSTU_latency_ratio_vs_SASRec",
    "p95_ms_sasrec": "SASRec_p95_ms",
    "p95_ms_hstu": "HSTU_p95_ms",
})

display(comp)

print("\nPRIMARY PAPER-SERVING VIEW — batch=1")
display(comp[comp["batch"] == 1].reset_index(drop=True))

print(
    "\nInterpretation: HSTU_speedup_vs_SASRec > 1 means HSTU is faster; "
    "< 1 means SASRec is faster."
)

In [ ]:
OUT = Path("/content/drive/MyDrive/hstu_pure_pytorch_ml1m")
sas_latency_df.to_csv(OUT / "sasrec_cached_a100_latency.csv", index=False)
comp.to_csv(OUT / "hstu_vs_sasrec_a100_latency.csv", index=False)
parity_df.to_csv(OUT / "sasrec_cached_parity.csv", index=False)

print("Saved:")
print(OUT / "sasrec_cached_a100_latency.csv")
print(OUT / "hstu_vs_sasrec_a100_latency.csv")
print(OUT / "sasrec_cached_parity.csv")